In [ ]:
library(dplyr)

data <- read.csv("../data/data.csv")
data <- as.matrix(data) # 转换为矩阵格式以便后续处理

# 剔除空值
data[is.nan(data)] <- NA
if (sum(is.na(data)) > 0) {
    print(paste(">>> 数据中存在", sum(is.na(data)), "个缺失值，已进行剔除处理。"))
    data <- na.omit(data)
}else {
    print(">>> 数据中不存在缺失值，无需剔除处理。")
}

# 提取 Y 和 X
Y <- data[, 1]
X <- data[, -1]
train_sd <- apply(X, 2, sd)
zero_var_cols <- which(train_sd == 0)
if (length(zero_var_cols) > 0) {
    cat(">>> 检测到", length(zero_var_cols), "个零方差特征，已剔除。\n")
    X <- X[, -zero_var_cols]
} else {
    cat(">>> 未检测到零方差特征，无需剔除。\n")
}

# 设定种子
set.seed(1111) 
n_total <- nrow(X)
train_idx <- sample(1:n_total, size = floor(0.7 * n_total))

X_train_raw <- X[train_idx, ]
Y_train     <- Y[train_idx]
X_test_raw  <- X[-train_idx, ]
Y_test      <- Y[-train_idx]

cat("训练集样本量:", nrow(X_train_raw), "| 测试集样本量:", nrow(X_test_raw), "\n")

# 1. 仅对训练集进行标准化，并提取均值(center)和标准差(scale)
X_train_scaled <- scale(X_train_raw)
train_centers  <- attr(X_train_scaled, "scaled:center")
train_scales   <- attr(X_train_scaled, "scaled:scale")

# 2. 强制使用训练集的参数对测试集进行转换！(绝对防御数据泄露)
X_test_scaled <- scale(X_test_raw, center = train_centers, scale = train_scales)

# 转回 data.frame 格式
X_train_scaled <- as.data.frame(X_train_scaled)
X_test_scaled  <- as.data.frame(X_test_scaled)

saveRDS(list(X = X_train_scaled, Y = Y_train), "../data/01_Train_Data.rds")
saveRDS(list(X = X_test_scaled, Y = Y_test), "../data/01_Test_Data.rds")